# Homework 2 - Training and Optimization

In [485]:
import tensorflow as tf
import numpy as np
import os
import zipfile
from glob import glob
from reader import AudioReader
from preprocessing import Padding, Normalization
from preprocessing import MelSpectrogram, MFCC

### Define the hyperparameters

In [487]:
SCRIPT_DIR = os.path.abspath('')

PREPROCESSING_ARGS = {
    'sampling_rate': 16000,
    'frame_length_in_s': 0.016,
    'frame_step_in_s': 0.016,
    'num_mel_bins': 30,
    'lower_frequency': 20,
    'upper_frequency': 6000,
    'num_coefficients': 30
}

TRAINING_ARGS = {
    'batch_size': 20,
    'learning_rate': 1.e-2,
    'end_learning_rate': 1.e-4,
    'epochs': 20,
    'width_multiplier': [0.25, 0.5, 0.75], # multipliers to use for structured pruning
}

### Create train/val/test Datasets and Callback

In [488]:
train_ds = tf.data.Dataset.list_files(['/tmp/msc-train/down*', '/tmp/msc-train/up*'])
val_ds = tf.data.Dataset.list_files(['/tmp/msc-val/down*', '/tmp/msc-val/up*'])
test_ds = tf.data.Dataset.list_files(['/tmp/msc-test/down*', '/tmp/msc-test/up*'])

In [489]:
# Learning Rate scheduler
linear_decay = tf.keras.optimizers.schedules.PolynomialDecay(
    initial_learning_rate=TRAINING_ARGS['learning_rate'],
    end_learning_rate=TRAINING_ARGS['end_learning_rate'],
    decay_steps=int(tf.data.experimental.cardinality(train_ds)/TRAINING_ARGS['batch_size']) * TRAINING_ARGS['epochs']
)

lr_scheduler = tf.keras.callbacks.LearningRateScheduler(linear_decay)

# Early Stopping
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=8,
    verbose=1,
    mode='min'
)

### Define the data pipeline

In [490]:
audio_reader = AudioReader(tf.int16)
padding = Padding(PREPROCESSING_ARGS['sampling_rate'])
normalization = Normalization(tf.int16)

if PREPROCESSING_ARGS['num_coefficients'] == 0:
    PREPROCESSING_ARGS.pop('num_coefficients')
    feature_processor = MelSpectrogram(**PREPROCESSING_ARGS)
    feature_processor_fn = feature_processor.get_mel_spec
    feature_processor_fn_lab = feature_processor.get_mel_spec_and_label
else:
    feature_processor = MFCC(**PREPROCESSING_ARGS)
    feature_processor_fn = feature_processor.get_mfccs
    feature_processor_fn_lab = feature_processor.get_mfccs_and_label

LABELS = ['down', 'up']

def prepare_for_training(feature, label):
    feature = tf.expand_dims(feature, -1)
    label_id = tf.argmax(label == LABELS)

    return feature, label_id

train_ds = (train_ds
            .map(audio_reader.get_audio_and_label)
            .map(padding.pad)
            .map(normalization.normalize)
            .map(feature_processor_fn_lab)
            .map(prepare_for_training)
            .batch(TRAINING_ARGS['batch_size'])
            .cache())
val_ds = (val_ds
            .map(audio_reader.get_audio_and_label)
            .map(padding.pad)
            .map(normalization.normalize)
            .map(feature_processor_fn_lab)
            .map(prepare_for_training)
            .batch(TRAINING_ARGS['batch_size']))
test_ds = (test_ds
            .map(audio_reader.get_audio_and_label)
            .map(padding.pad)
            .map(normalization.normalize)
            .map(feature_processor_fn_lab)
            .map(prepare_for_training)
            .batch(TRAINING_ARGS['batch_size']))

### Define the data pipeline 

### Read a batch of data

In [491]:
for example_batch, example_labels in train_ds.take(1):
  print('Batch Shape:', example_batch.shape)
  print('Data Shape:', example_batch.shape[1:])
  print('Labels:', example_labels)
     

Batch Shape: (20, 62, 30, 1)
Data Shape: (62, 30, 1)
Labels: tf.Tensor([1 0 0 0 0 1 1 1 1 1 0 1 0 1 0 0 0 0 1 1], shape=(20,), dtype=int64)


2024-12-20 19:09:49.925221: W tensorflow/core/kernels/data/cache_dataset_ops.cc:858] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


### Create the model

In [492]:
wm = TRAINING_ARGS['width_multiplier']

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=example_batch.shape[1:]),
    tf.keras.layers.Conv2D(filters=int(128*wm[1]), kernel_size=[3, 3], strides=[2, 2], use_bias=False, padding='valid'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.DepthwiseConv2D(kernel_size=[3, 3], strides=[1, 1], use_bias=False, padding='same'),
    tf.keras.layers.Conv2D(filters=int(128*wm[1]), kernel_size=[1, 1], strides=[1, 1], use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.DepthwiseConv2D(kernel_size=[5, 5], strides=[1, 1], use_bias=False, padding='same'),
    tf.keras.layers.Conv2D(filters=int(128*wm[1]), kernel_size=[1, 1], strides=[1, 1], use_bias=False),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU(),
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dense(units=len(LABELS)),
    tf.keras.layers.Softmax()
])

In [493]:
model.summary()

Model: "sequential_28"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d_84 (Conv2D)          (None, 30, 14, 64)        576       
                                                                 
 batch_normalization_84 (Ba  (None, 30, 14, 64)        256       
 tchNormalization)                                               
                                                                 
 re_lu_84 (ReLU)             (None, 30, 14, 64)        0         
                                                                 
 depthwise_conv2d_52 (Depth  (None, 30, 14, 64)        576       
 wiseConv2D)                                                     
                                                                 
 conv2d_85 (Conv2D)          (None, 30, 14, 64)        4096      
                                                                 
 batch_normalization_85 (Ba  (None, 30, 14, 64)      

### Train the model with callback

In [494]:
loss = tf.losses.SparseCategoricalCrossentropy(from_logits=False)
optimizer = tf.optimizers.Adam(learning_rate=linear_decay)
metrics = [tf.metrics.SparseCategoricalAccuracy()]
model.compile(loss=loss, optimizer=optimizer, metrics=metrics)

history = model.fit(
    train_ds, 
    epochs=TRAINING_ARGS['epochs'], 
    validation_data=val_ds, 
    callbacks=[lr_scheduler, early_stopping]
)

Epoch 1/20
80/80 [==============================] - 3s 14ms/step - loss: 0.4440 - sparse_categorical_accuracy: 0.8056 - val_loss: 0.4005 - val_sparse_categorical_accuracy: 0.8350 - lr: 0.0095
Epoch 2/20
80/80 [==============================] - 1s 12ms/step - loss: 0.2991 - sparse_categorical_accuracy: 0.8888 - val_loss: 0.2743 - val_sparse_categorical_accuracy: 0.8950 - lr: 0.0090
Epoch 3/20
80/80 [==============================] - 1s 12ms/step - loss: 0.2133 - sparse_categorical_accuracy: 0.9325 - val_loss: 0.2193 - val_sparse_categorical_accuracy: 0.9000 - lr: 0.0085
Epoch 4/20
80/80 [==============================] - 1s 13ms/step - loss: 0.1540 - sparse_categorical_accuracy: 0.9525 - val_loss: 0.1362 - val_sparse_categorical_accuracy: 0.9400 - lr: 0.0080
Epoch 5/20
80/80 [==============================] - 1s 14ms/step - loss: 0.1410 - sparse_categorical_accuracy: 0.9463 - val_loss: 0.1076 - val_sparse_categorical_accuracy: 0.9600 - lr: 0.0075
Epoch 6/20
80/80 [======================

### Show the history

In [495]:
history.history

{'loss': [0.44404342770576477,
  0.2990759015083313,
  0.2133387178182602,
  0.15402159094810486,
  0.14102701842784882,
  0.10751598328351974,
  0.09145235270261765,
  0.07879500091075897,
  0.07273770868778229,
  0.055678728967905045,
  0.04744675010442734,
  0.040345802903175354,
  0.03469889983534813,
  0.02907240204513073,
  0.024433696642518044,
  0.02224728651344776,
  0.018188169226050377,
  0.016081858426332474,
  0.014366375282406807,
  0.012923014350235462],
 'sparse_categorical_accuracy': [0.8056250214576721,
  0.8887500166893005,
  0.9325000047683716,
  0.9524999856948853,
  0.9462500214576721,
  0.9618750214576721,
  0.96875,
  0.9725000262260437,
  0.9775000214576721,
  0.9831249713897705,
  0.9818750023841858,
  0.9868749976158142,
  0.9881250262260437,
  0.9906250238418579,
  0.9918749928474426,
  0.9925000071525574,
  0.9931250214576721,
  0.9943749904632568,
  0.9956250190734863,
  0.9962499737739563],
 'val_loss': [0.40045279264450073,
  0.2743152379989624,
  0.2192

### Evaluate the model

In [496]:
training_loss = history.history['loss'][-1]
training_accuracy = history.history['sparse_categorical_accuracy'][-1]
val_loss = history.history['val_loss'][-1]
val_accuracy = history.history['val_sparse_categorical_accuracy'][-1]

test_loss, test_accuracy = model.evaluate(test_ds)

print(f'Training Loss: {training_loss:.4f}')
print(f'Training Accuracy: {training_accuracy*100.:.2f}%')
print()
print(f'Validation Loss: {val_loss:.4f}')
print(f'Validation Accuracy: {val_accuracy*100.:.2f}%')
print()
print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy*100.:.2f}%')

10/10 [==============================] - 0s 6ms/step - loss: 0.0310 - sparse_categorical_accuracy: 0.9950
Training Loss: 0.0129
Training Accuracy: 99.62%

Validation Loss: 0.0389
Validation Accuracy: 98.50%

Test Loss: 0.0310
Test Accuracy: 99.50%


### Save model and hyperparameters

In [497]:
import os
from time import time

timestamp = int(time())

saved_model_dir = f'./saved_models/{timestamp}'
if not os.path.exists(saved_model_dir):
    os.makedirs(saved_model_dir)
model.save(saved_model_dir)

INFO:tensorflow:Assets written to: ./saved_models/1734718213/assets


INFO:tensorflow:Assets written to: ./saved_models/1734718213/assets


In [498]:
import pandas as pd

output_dict = {
    'timestamp': timestamp,
    **PREPROCESSING_ARGS,
    **TRAINING_ARGS,
    'test_accuracy': test_accuracy
}

df = pd.DataFrame([output_dict])

output_path='./mel_spectrogram_results.csv'
df.to_csv(output_path, mode='a', header=not os.path.exists(output_path), index=False)

### TFLite Conversion

In [499]:
converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
tflite_model = converter.convert()

tflite_model_dir = './tflite_models'
if not os.path.exists(tflite_model_dir):
    os.makedirs(tflite_model_dir)

tflite_model_name = os.path.join(tflite_model_dir, f'{timestamp}.tflite')
tflite_model_name

with open(tflite_model_name, 'wb') as fp:
    fp.write(tflite_model)

2024-12-20 19:10:15.817863: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2024-12-20 19:10:15.817895: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2024-12-20 19:10:15.818127: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: ./saved_models/1734718213
2024-12-20 19:10:15.821817: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2024-12-20 19:10:15.821849: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: ./saved_models/1734718213
2024-12-20 19:10:15.831560: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2024-12-20 19:10:15.916625: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: ./saved_models/1734718213
2024-12-20 19:10:15.948438: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status

### Save and Evaluate TFLite Model

In [503]:
MODEL_FILE_PATH = tflite_model_name

model_size = os.path.getsize(MODEL_FILE_PATH)

if MODEL_FILE_PATH.endswith('.zip'):
    with zipfile.ZipFile(MODEL_FILE_PATH, 'r') as fp:
        fp.extractall('/tmp/')
        model_filename = fp.namelist()[0]
        MODEL_FILE_PATH = '/tmp/' + model_filename

interpreter = tf.lite.Interpreter(model_path=MODEL_FILE_PATH)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Number of inputs:", len(input_details))
print("Number of outputs:", len(output_details))
print("Input name:", input_details[0]['name'])
print("Input shape:", input_details[0]['shape'])
print("Output name:", output_details[0]['name'])
print("Output shape:", output_details[0]['shape'])

Number of inputs: 1
Number of outputs: 1
Input name: serving_default_input_29:0
Input shape: [ 1 62 30  1]
Output name: StatefulPartitionedCall:0
Output shape: [1 2]


In [504]:
filenames = glob('/tmp/msc-test/down*') + glob('/tmp/msc-test/up*')

accuracy = 0.0

for filename in filenames:
    audio, true_label = audio_reader.get_audio_and_label(filename)   
    true_label = true_label.numpy().decode()
    
    audio = padding.pad_audio(audio)
    audio = normalization.normalize_audio(audio)
    features = feature_processor_fn(audio)
    features = tf.expand_dims(features, 0)
    features = tf.expand_dims(features, -1)

    interpreter.set_tensor(input_details[0]['index'], features)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details[0]['index'])

    top_index = np.argmax(output[0])
    predicted_label = LABELS[top_index]

    accuracy += true_label == predicted_label

accuracy /= len(filenames)

In [505]:
print(f'Accuracy: {100 * accuracy:.3f}%')
print(f'Model size: {model_size / 2 ** 10:.1f}KB')

Accuracy: 99.500%
Model size: 48.2KB


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=b4ef5aa4-3f71-4837-91f1-c6fd9810a7ea' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>